In [1]:
import os, re, json, time, warnings
import numpy as np
import pandas as pd
import torch
import jieba
from torch.utils.data import Dataset
from openai import OpenAI
from sklearn.model_selection import train_test_split
from transformers import (
    BertTokenizerFast,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline as hf_pipeline,
)
import evaluate as hf_evaluate

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════
#  ① 配置区（只需修改这里）
# ══════════════════════════════════════════════
DEEPSEEK_API_KEY = "sk-abbc003b0d894dec939af36621a17d58"                   # ← DeepSeek API Key
ANNOTATION_FILE  = r"E:\00论文\2025江苏人民公社\JAABE返修1\副本annotation_template.xlsx" # ← 标注文件（和脚本放同一文件夹）
OUTPUT_FILE      = r"E:\00论文\2025江苏人民公社\JAABE返修1\ner_comparison_results.xlsx"
BERT_SAVE_PATH   = "./commune_ner_model"           # 微调模型保存路径

# BERT 微调超参
BASE_MODEL  = r"E:\05code\bert-base-chinese"   # ← 本地模型路径，无需联网
MAX_LEN     = 128
EPOCHS      = 15
BATCH_SIZE  = 8
LR          = 2e-5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备：{DEVICE}\n")


c:\anaconda3\envs\ner_env\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
c:\anaconda3\envs\ner_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


使用设备：cuda



In [2]:
# ══════════════════════════════════════════════
#  ③ 公共工具：同义词归并 / 停用词 / 指标计算
# ══════════════════════════════════════════════
SYNONYM_MAP = {
    "养猪": "猪", "生猪": "猪",
    "养蚕": "蚕", "养禽": "禽", "养鸡": "禽", "养鸭": "禽", "鹅鸟": "禽",
    "养鱼": "鱼", "养牛": "牛", "养羊": "羊", "养兔": "兔",
    "化肥工场": "化肥工厂", "工场": "工厂",
    "水稻": "米", "稻谷": "米", "中稻": "米", "晚稻": "米", "季节性中稻": "米",
    "麦类": "麦", "麦子": "麦",
    "牧畜": "畜牧",
}

STOPWORDS = {
    "的","了","是","在","也","和","与","或","及","有","为","将","被",
    "年","月","日","个","亩","斤","元","户","人","头","株","所","件","种",
    "等","中","上","下","内","外","前","后","间","处",
    "比","约","余","多","共","总","每","全",
    "生产","进行","开展","增加","建设","管理","利用","采用",
    "地区","面积","种植","播种","收入","情况","活动","整备","储藏",
    "社员","大队","生产队","小队","公社","危害","粪便",
    "季节性","业余","文化","技术","学校","学习班","修配",
}

# 需要保留的单字实体
SINGLE_CHAR_ENTITIES = {"米","麦","绵","粟","麻","茶","桑","竹","豆"}

def normalize(entities: set) -> set:
    """同义词归并，与 gold 标注风格保持一致"""
    return set(SYNONYM_MAP.get(e.strip(), e.strip()) for e in entities if e.strip())

def parse_gold(s: str) -> set:
    return normalize(set(str(s).split()))

def compute_metrics(preds: list, golds: list) -> dict:
    tp = fp = fn = 0
    for pred, gold in zip(preds, golds):
        tp += len(pred & gold)
        fp += len(pred - gold)
        fn += len(gold - pred)
    p = tp / (tp + fp + 1e-9)
    r = tp / (tp + fn + 1e-9)
    f = 2 * p * r / (p + r + 1e-9)
    return {
        "Precision": round(p, 4),
        "Recall":    round(r, 4),
        "F1":        round(f, 4),
        "TP": tp, "FP": fp, "FN": fn,
    }


In [4]:
#  ③ 方法一：Jieba 分词
# ══════════════════════════════════════════════
DOMAIN_WORDS = [
    "养猪","养蚕","养禽","养鱼","养牛","养羊","养兔",
    "绿肥","化肥","农机具","农具","水利","灌溉","蔬菜",
    "渔业","畜牧","副业","手工业","加工厂","修配站",
    "油菜籽","番薯","杂谷","玉米","烟草","药材","薄荷",
    "黄麻","豌豆","竹器","编织物","粮食作物","牧畜","野菜",
]
for w in DOMAIN_WORDS:
    jieba.add_word(w)

def jieba_ner(text: str) -> set:
    words = jieba.lcut(str(text))
    ents = set()
    for w in words:
        w = w.strip()
        if w in SINGLE_CHAR_ENTITIES:
            ents.add(w)
            continue
        if (w and w not in STOPWORDS
                and not re.fullmatch(r"[\d\s\W]+", w)
                and len(w) >= 2):
            ents.add(w)
    return normalize(ents)

# ══════════════════════════════════════════════
#  ④ 方法二：微调 BERT + pipeline
# ══════════════════════════════════════════════
LABEL2ID = {"O": 0, "B-ENT": 1, "I-ENT": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def build_bio_labels(text: str, entity_words: list) -> list:
    labels = [0] * len(text)
    used   = [False] * len(text)
    for word in sorted(entity_words, key=len, reverse=True):
        start = 0
        while True:
            pos = text.find(word, start)
            if pos == -1:
                break
            if not any(used[pos:pos + len(word)]):
                labels[pos] = 1
                for i in range(1, len(word)):
                    labels[pos + i] = 2
                for i in range(len(word)):
                    used[pos + i] = True
            start = pos + 1
    return labels

class CommuneNERDataset(Dataset):
    def __init__(self, texts, gold_sets, tokenizer):
        self.samples = []
        for text, gold in zip(texts, gold_sets):
            char_labels = build_bio_labels(text, list(gold))
            if 1 not in char_labels:
                continue
            enc = tokenizer(
                text,
                max_length=MAX_LEN,
                padding="max_length",
                truncation=True,
                return_offsets_mapping=True,
                return_tensors="pt",
            )
            offsets = enc["offset_mapping"][0].tolist()
            padded  = char_labels + [-100] * MAX_LEN
            token_labels = [-100 if s == e else padded[s] for s, e in offsets]
            self.samples.append({
                "input_ids":      enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "token_type_ids": enc["token_type_ids"].squeeze(0),
                "labels":         torch.tensor(token_labels, dtype=torch.long),
            })

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]

def _hf_compute_metrics(p):
    """离线版评估函数，直接用 seqeval 库"""
    from seqeval.metrics import precision_score, recall_score, f1_score
    preds  = np.argmax(p.predictions, axis=2)
    labels = p.label_ids
    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        tp, tl = [], []
        for p_id, l_id in zip(pred_seq, label_seq):
            if l_id == -100:
                continue
            tp.append(ID2LABEL[p_id])
            tl.append(ID2LABEL[l_id])
        true_preds.append(tp)
        true_labels.append(tl)
    return {
        "precision": round(precision_score(true_labels, true_preds), 4),
        "recall":    round(recall_score(true_labels, true_preds),    4),
        "f1":        round(f1_score(true_labels, true_preds),        4),
    }

def finetune_bert(texts, gold_sets):
    print(f"\n[BERT] 加载 tokenizer：{BASE_MODEL}")
    tokenizer = BertTokenizerFast.from_pretrained(BASE_MODEL)

    tr_texts, val_texts, tr_golds, val_golds = train_test_split(
        texts, gold_sets, test_size=0.2, random_state=42
    )
    print(f"[BERT] 训练集 {len(tr_texts)} 条，验证集 {len(val_texts)} 条")

    train_ds = CommuneNERDataset(tr_texts, tr_golds, tokenizer)
    val_ds   = CommuneNERDataset(val_texts, val_golds, tokenizer)
    print(f"[BERT] 有效训练样本：{len(train_ds)}，有效验证样本：{len(val_ds)}")

    print(f"[BERT] 加载本地基础模型：{BASE_MODEL}")
    model = AutoModelForTokenClassification.from_pretrained(
        BASE_MODEL,
        num_labels=len(LABEL2ID),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    args = TrainingArguments(
        output_dir=BERT_SAVE_PATH,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR,
        weight_decay=0.01,
        warmup_ratio=0.1,
        evaluation_strategy="epoch",   # transformers 4.36 用这个参数名
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=5,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=_hf_compute_metrics,
    )
    print("[BERT] 开始微调...")
    trainer.train()
    trainer.save_model(BERT_SAVE_PATH)
    tokenizer.save_pretrained(BERT_SAVE_PATH)
    print(f"[BERT] 微调完成，模型已保存：{BERT_SAVE_PATH}")

def load_bert_pipeline():
    pipe = hf_pipeline(
        "ner",
        model=BERT_SAVE_PATH,
        tokenizer=BERT_SAVE_PATH,
        aggregation_strategy="simple",
        device=0 if torch.cuda.is_available() else -1,
    )
    print(f"[BERT] pipeline 加载成功：{BERT_SAVE_PATH}")
    return pipe

def bert_ner(text: str, pipe) -> set:
    result = pipe(str(text))
    raw = {
        r["word"].replace(" ", "").replace("##", "")
        for r in result if r["entity_group"] == "ENT"
    }
    return normalize(raw)

# ══════════════════════════════════════════════
#  ⑤ 方法三：DeepSeek LLM
# ══════════════════════════════════════════════
client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

SYSTEM_PROMPT = """你是处理中国人民公社（1958-1983年）历史文献的命名实体识别专家。
从给定文本中提取所有"生产活动实体"，包括：
- 农业产品（米、麦、玉米、蔬菜、油菜籽、番薯等，包括单字词）
- 养殖活动（统一用动物名：养猪→猪，养蚕→蚕，养禽→禽）
- 工业/副业（化肥工厂、手工业、农机具、竹器等）
- 其他生产资源（绿肥、水利等）

规则：
1. 只返回JSON数组，如 ["米", "猪", "蔬菜"]
2. 用核心词，不含数量、年份、面积等修饰词
3. 忽略行政管理词（生产大队、社员、公社等）
4. 没有实体返回 []

示例：
输入：麦类种植面积11000亩；麦、米、养猪
输出：["麦", "米", "猪"]

输入：1.储藏粪便的地区3900余所 2.麦
输出：["麦"]

输入：1.铁、木、竹、瓦、铜匠243人 2.副业（竹器等），绵
输出：["竹器", "绵"]

输入：生产大队23、社员5107户；麦、米、养猪、59年13868头
输出：["麦", "米", "猪"]
"""

def llm_ner(text: str, retry: int = 3) -> set:
    for attempt in range(retry):
        try:
            resp = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": str(text)},
                ],
                temperature=0.0,
                max_tokens=200,
            )
            raw = resp.choices[0].message.content.strip()
            raw = re.sub(r"```json|```", "", raw).strip()
            return normalize(set(json.loads(raw)))
        except json.JSONDecodeError:
            matches = re.findall(r'"([^"]+)"', raw)
            if matches:
                return normalize(set(matches))
            time.sleep(1)
        except Exception as e:
            print(f"    API错误({attempt+1}): {e}")
            time.sleep(2)
    return set()

# ══════════════════════════════════════════════
#  ⑥ 主流程
# ══════════════════════════════════════════════
# 读取标注数据
print("=" * 55)
print("读取标注文件...")
ann = pd.read_excel(ANNOTATION_FILE)
ann = ann[
    ann["gold_entities"].notna() &
    (ann["gold_entities"].astype(str).str.strip() != "")
].reset_index(drop=True)
print(f"有效标注样本：{len(ann)} 条")

texts = ann["COL_G"].astype(str).tolist()
golds = [parse_gold(r["gold_entities"]) for _, r in ann.iterrows()]

# BERT 微调（如果已存在则跳过）
print("=" * 55)
if os.path.exists(os.path.join(BERT_SAVE_PATH, "config.json")):
    print(f"[BERT] 检测到已有微调模型，跳过训练：{BERT_SAVE_PATH}")
else:
    finetune_bert(texts, golds)

bert_pipe = load_bert_pipeline()

# 三方法推理
print("\n" + "=" * 55)
print("[1/3] Jieba 推理...")
jieba_preds = [jieba_ner(t) for t in texts]
jm = compute_metrics(jieba_preds, golds)
print(f"      P={jm['Precision']}  R={jm['Recall']}  F1={jm['F1']}")

print("\n[2/3] 微调BERT 推理...")
bert_preds = [bert_ner(t, bert_pipe) for t in texts]
bm = compute_metrics(bert_preds, golds)
print(f"      P={bm['Precision']}  R={bm['Recall']}  F1={bm['F1']}")

print("\n[3/3] DeepSeek LLM 推理（每条间隔0.3s）...")
llm_preds = []
for i, t in enumerate(texts):
    print(f"      {i+1}/{len(texts)}: {t[:40]}")
    llm_preds.append(llm_ner(t))
    time.sleep(0.3)
lm = compute_metrics(llm_preds, golds)
print(f"      P={lm['Precision']}  R={lm['Recall']}  F1={lm['F1']}")

# 汇总对比表
metrics_df = pd.DataFrame([
    {"方法": "Jieba分词",          **jm},
    {"方法": "微调BERT",           **bm},
    {"方法": "DeepSeek LLM",       **lm},
])

print("\n" + "=" * 55)
print("         NER 三方法性能对比")
print("=" * 55)
print(metrics_df[["方法", "Precision", "Recall", "F1"]].to_string(index=False))
print("=" * 55)

# 逐条详情
detail_rows = []
for i, (text, gold) in enumerate(zip(texts, golds)):
    detail_rows.append({
        "原文":         text,
        "黄金标准":     " ".join(sorted(gold)),
        "Jieba结果":    " ".join(sorted(jieba_preds[i])),
        "Jieba_正确":   " ".join(sorted(jieba_preds[i] & gold)),
        "Jieba_多识别": " ".join(sorted(jieba_preds[i] - gold)),
        "Jieba_漏识别": " ".join(sorted(gold - jieba_preds[i])),
        "BERT结果":     " ".join(sorted(bert_preds[i])),
        "BERT_正确":    " ".join(sorted(bert_preds[i] & gold)),
        "BERT_多识别":  " ".join(sorted(bert_preds[i] - gold)),
        "BERT_漏识别":  " ".join(sorted(gold - bert_preds[i])),
        "LLM结果":      " ".join(sorted(llm_preds[i])),
        "LLM_正确":     " ".join(sorted(llm_preds[i] & gold)),
        "LLM_多识别":   " ".join(sorted(llm_preds[i] - gold)),
        "LLM_漏识别":   " ".join(sorted(gold - llm_preds[i])),
    })
detail_df = pd.DataFrame(detail_rows)

# 写出 Excel
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="性能对比", index=False)
    detail_df.to_excel(writer, sheet_name="逐条详情", index=False)

print(f"\n结果已写入：{OUTPUT_FILE}")
print("  Sheet「性能对比」：Precision / Recall / F1 / TP / FP / FN")
print("  Sheet「逐条详情」：每条的正确 / 多识别 / 漏识别明细")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at E:\05code\bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


读取标注文件...
有效标注样本：49 条

[BERT] 加载 tokenizer：E:\05code\bert-base-chinese
[BERT] 训练集 39 条，验证集 10 条
[BERT] 有效训练样本：38，有效验证样本：10
[BERT] 加载本地基础模型：E:\05code\bert-base-chinese
[BERT] 开始微调...



You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.





                                              

                                        
  7%|▋         | 5/75 [22:58<00:12,  5.61it/s]


{'loss': 1.3268, 'learning_rate': 1.25e-05, 'epoch': 1.0}




                                              

                                        


                            
  7%|▋         | 5/75 [22:58<00:12,  5.61it/s]






...

{'loss': 0.0067, 'learning_rate': 0.0, 'epoch': 15.0}




                                              

                                         
                                           


  7%|▋         | 5/75 [23:26<00:12,  5.61it/s]






{'eval_loss': 0.21675792336463928, 'eval_precision': 0.8182, 'eval_recall': 0.931, 'eval_f1': 0.871, 'eval_runtime': 0.0483, 'eval_samples_per_second': 207.235, 'eval_steps_per_second': 41.447, 'epoch': 15.0}


                                              

                                         
100%|██████████| 75/75 [00:29<00:00,  2.51it/s]


{'train_runtime': 29.8719, 'train_samples_per_second': 19.082, 'train_steps_per_second': 2.511, 'train_loss': 0.25070585911472637, 'epoch': 15.0}
[BERT] 微调完成，模型已保存：./commune_ner_model
[BERT] pipeline 加载成功：./commune_ner_model

[1/3] Jieba 推理...
      P=0.5749  R=0.9412  F1=0.7138

[2/3] 微调BERT 推理...
      P=0.9524  R=0.9804  F1=0.9662

[3/3] DeepSeek LLM 推理（每条间隔0.3s）...
      1/49: 渔业，大麦、米
      2/49: 2.米
      3/49: 2.麦
      4/49: 1.储藏粪便的地区3900余所 2.麦
      5/49: 农具整备活动
      6/49: 1.化肥工场
      7/49: 2.米
      8/49: 1.农具工厂
      9/49: 生产院57；麦、绿肥
      10/49: 2.蔬菜
      11/49: 麦类种植面积11000亩；麦、米、养猪
      12/49: 2.养猪
      13/49: 1.小麦播种面积50000亩 2.小麦
      14/49: 1.绵播种面积5000亩，农机具修配站5个 2.米、绵
      15/49: 蔬菜（白菜、萝卜、黄瓜等）
      16/49: 2.绵
      17/49: 1.200多个生产队 2.米
      18/49: 杂谷种植面积33000亩、杂谷（60年总收入高达16920000斤）、番薯、小麦
      19/49: 2.养猪
      20/49: 圩塘生产大队：生产队18、米
      21/49: 小麦
      22/49: 生产大队13、竹木农机具修配站；米
      23/49: 米
      24/49: 2.米
      25/49: 米
      26/49: 米、蔬菜
      27/49: 生产大队23、社

In [6]:
"""
用微调好的 BERT 对全量 COL_G 列进行命名实体识别
输出结果写入新的 Excel 文件

直接在 Jupyter 运行，或命令行：python ner_inference.py
"""

import re
import pandas as pd
import torch
from transformers import pipeline as hf_pipeline

# ══════════════════════════════════════════════
#  配置（只需修改这里）
# ══════════════════════════════════════════════
INPUT_FILE   = r"E:\00论文\2025江苏人民公社\people's commune.xlsx"
OUTPUT_FILE  = r"E:\00论文\2025江苏人民公社\people's commune_NER结果.xlsx"
BERT_MODEL   = r"E:\00论文\2025江苏人民公社\【submit1】heritage science\0602analysis\commune_ner_model"  # 微调好的模型路径
TARGET_COL   = "COL_G"

# ══════════════════════════════════════════════
#  同义词归并（与标注风格一致）
# ══════════════════════════════════════════════
SYNONYM_MAP = {
    "养猪": "猪", "生猪": "猪",
    "养蚕": "蚕", "养禽": "禽", "养鸡": "禽", "养鸭": "禽", "鹅鸟": "禽",
    "养鱼": "鱼", "养牛": "牛", "养羊": "羊", "养兔": "兔",
    "化肥工场": "化肥工厂", "工场": "工厂",
    "水稻": "米", "稻谷": "米", "中稻": "米", "晚稻": "米", "季节性中稻": "米",
    "麦类": "麦", "麦子": "麦",
    "牧畜": "畜牧",
}

def normalize(entities: set) -> set:
    return set(SYNONYM_MAP.get(e.strip(), e.strip()) for e in entities if e.strip())

def bert_ner(text: str, pipe) -> set:
    """对单条文本做 NER，返回归并后的实体集合"""
    text = str(text).strip()
    if not text:
        return set()
    result = pipe(text)
    raw = {
        r["word"].replace(" ", "").replace("##", "")
        for r in result if r["entity_group"] == "ENT"
    }
    return normalize(raw)

# ══════════════════════════════════════════════
#  主流程
# ══════════════════════════════════════════════

# 1. 加载模型
print("加载微调BERT模型...")
device = 0 if torch.cuda.is_available() else -1
print(f"使用设备：{'GPU' if device == 0 else 'CPU'}")

pipe = hf_pipeline(
    "ner",
    model=BERT_MODEL,
    tokenizer=BERT_MODEL,
    aggregation_strategy="simple",
    device=device,
)
print("模型加载成功！\n")

# 2. 读取数据
print(f"读取文件：{INPUT_FILE}")
df = pd.read_excel(INPUT_FILE)
print(f"总行数：{len(df)}，{TARGET_COL} 非空：{df[TARGET_COL].notna().sum()} 条\n")

# 3. 逐行推理
print("开始命名实体识别...")
ner_results     = []   # 识别出的实体（分号分隔字符串）
ner_entity_list = []   # 实体列表（逗号分隔，便于后续统计）
ner_count       = []   # 实体数量

for i, row in df.iterrows():
    raw = row[TARGET_COL]

    if pd.isna(raw) or str(raw).strip() == "":
        ner_results.append("")
        ner_entity_list.append("")
        ner_count.append(0)
        continue

    entities = bert_ner(raw, pipe)

    ner_results.append("；".join(sorted(entities)))
    ner_entity_list.append(",".join(sorted(entities)))
    ner_count.append(len(entities))

    # 每50条打印一次进度
    if (i + 1) % 50 == 0 or (i + 1) == len(df):
        print(f"  进度：{i+1}/{len(df)}")

print(f"\n识别完成！共处理 {len(df)} 条，识别到实体 {sum(ner_count)} 个\n")

# 4. 写回 DataFrame
df["NER_实体"]   = ner_results      # 主要结果列：实体用分号分隔
df["NER_列表"]   = ner_entity_list  # 逗号分隔版本，便于后续用 split(',') 统计
df["NER_数量"]   = ner_count        # 每行识别出的实体数量

# 5. 输出 Excel
df.to_excel(OUTPUT_FILE, index=False)
print(f"结果已写入：{OUTPUT_FILE}")

# 6. 打印统计摘要
print("\n─── 识别结果摘要 ───")
print(f"有实体的行数：{(df['NER_数量'] > 0).sum()}")
print(f"无实体的行数：{(df['NER_数量'] == 0).sum()}")
print(f"平均每行实体数：{df[df['NER_数量']>0]['NER_数量'].mean():.2f}")

# 统计最常见的实体
from collections import Counter
all_entities = []
for ents in df["NER_列表"]:
    if ents:
        all_entities.extend([e.strip() for e in str(ents).split(",") if e.strip()])

print("\n出现最多的前20个实体：")
for ent, cnt in Counter(all_entities).most_common(20):
    print(f"  {ent}: {cnt} 次")

# 7. 预览前10行结果
print("\n─── 前10行结果预览 ───")
preview = df[[TARGET_COL, "NER_实体", "NER_数量"]].head(10).fillna("")
print(preview.to_string(index=False))


加载微调BERT模型...
使用设备：GPU
模型加载成功！

读取文件：E:\00论文\2025江苏人民公社\people's commune.xlsx
总行数：334，COL_G 非空：284 条

开始命名实体识别...
  进度：50/334
  进度：100/334
  进度：150/334
  进度：200/334
  进度：250/334
  进度：300/334
  进度：334/334

识别完成！共处理 334 条，识别到实体 664 个

结果已写入：E:\00论文\2025江苏人民公社\people's commune_NER结果.xlsx

─── 识别结果摘要 ───
有实体的行数：277
无实体的行数：57
平均每行实体数：2.40

出现最多的前20个实体：
  米: 105 次
  麦: 52 次
  小麦: 36 次
  猪: 28 次
  绵: 27 次
  玉米: 23 次
  蔬菜: 22 次
  番薯: 16 次
  大豆: 15 次
  大麦: 14 次
  鱼: 7 次
  蚕: 7 次
  绿肥: 7 次
  高粱: 6 次
  棉花: 6 次
  红薯: 6 次
  菜: 6 次
  河泥: 5 次
  油菜籽: 5 次
  农具工厂: 5 次

─── 前10行结果预览 ───
        COL_G   NER_实体  NER_数量
           肥料       肥料       1
  养猪、蔬菜、养鱼、养禽 猪；禽；蔬菜；鱼       4
       收音机加工厂   工厂；收音机       2
           安培                0
            伞        伞       1
蔬菜（白菜、萝卜、黄瓜等） 白菜；萝卜；黄瓜       3
        发电机六台      发电机       1
                             0
                             0
                             0


加载微调BERT模型...
使用设备：GPU
模型加载成功！

读取文件：E:\00论文\2025江苏人民公社\people's commune.xlsx
总行数：334，COL_G 非空：284 条

开始命名实体识别...
  进度：50/334
  进度：100/334
  进度：150/334
  进度：200/334
  进度：250/334
  进度：300/334
  进度：334/334

识别完成！共处理 334 条，识别到实体 664 个

结果已写入：E:\00论文\2025江苏人民公社\people's commune_NER结果.xlsx

─── 识别结果摘要 ───
有实体的行数：277
无实体的行数：57
平均每行实体数：2.40

出现最多的前20个实体：
  米: 105 次
  麦: 52 次
  小麦: 36 次
  猪: 28 次
  绵: 27 次
  玉米: 23 次
  蔬菜: 22 次
  番薯: 16 次
  大豆: 15 次
  大麦: 14 次
  鱼: 7 次
  蚕: 7 次
  绿肥: 7 次
  高粱: 6 次
  棉花: 6 次
  红薯: 6 次
  菜: 6 次
  河泥: 5 次
  油菜籽: 5 次
  农具工厂: 5 次

─── 前10行结果预览 ───
        COL_G   NER_实体  NER_数量
           肥料       肥料       1
  养猪、蔬菜、养鱼、养禽 猪；禽；蔬菜；鱼       4
       收音机加工厂   工厂；收音机       2
           安培                0
            伞        伞       1
蔬菜（白菜、萝卜、黄瓜等） 白菜；萝卜；黄瓜       3
        发电机六台      发电机       1
                             0
                             0
                             0

In [7]:
"""
基于 NER 识别结果计算 TF-IDF
- 文档（Document）= 每个公社（每行）
- 词（Term）       = NER 识别出的实体词
- 输出：
    ① tfidf_matrix.xlsx   每个公社×每个实体的TF-IDF矩阵
    ② tfidf_summary.xlsx  每个实体的平均TF-IDF、总频次、文档频次、IDF值
    ③ 终端打印Top20实体
"""

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import math

# ══════════════════════════════════════════════
# 配置
# ══════════════════════════════════════════════
INPUT_FILE    = r"E:\00论文\2025江苏人民公社\people's commune_NER结果.xlsx"
OUTPUT_MATRIX = r"E:\00论文\2025江苏人民公社\tfidf_matrix.xlsx"
OUTPUT_SUMMARY= r"E:\00论文\2025江苏人民公社\tfidf_summary.xlsx"

# ══════════════════════════════════════════════
# 1. 读取数据
# ══════════════════════════════════════════════
print("读取文件...")
df = pd.read_excel(INPUT_FILE)
print(f"总行数：{len(df)}，NER_实体非空：{df['NER_实体'].notna().sum()} 行")

# ══════════════════════════════════════════════
# 2. 构建"文档"列表
#    每个公社的实体词用空格连接，作为一个"文档"
#    空行填充为空字符串
# ══════════════════════════════════════════════
def parse_entities(s):
    """将 '猪；禽；蔬菜；鱼' 解析为 ['猪', '禽', '蔬菜', '鱼']"""
    if pd.isna(s) or str(s).strip() == '':
        return []
    return [e.strip() for e in str(s).split('；') if e.strip()]

df['实体列表'] = df['NER_实体'].apply(parse_entities)

# 用空格连接成字符串（TfidfVectorizer 用空格分词）
# 注意：把多字实体里的空格替换掉，避免被错误切分
df['文档文本'] = df['实体列表'].apply(
    lambda lst: ' '.join([e.replace(' ', '_') for e in lst])
)

# 过滤掉空文档
valid_mask = df['文档文本'].str.strip() != ''
print(f"有效公社（含实体）：{valid_mask.sum()} 个")

# ══════════════════════════════════════════════
# 3. 计算 TF-IDF
# ══════════════════════════════════════════════
print("\n计算 TF-IDF...")

vectorizer = TfidfVectorizer(
    token_pattern=r'(?u)\S+',   # 以空格分词，支持中文词语
    smooth_idf=True,             # IDF平滑：log((N+1)/(df+1))+1，避免除零
    sublinear_tf=False,          # 不对TF取对数（保留原始词频比例）
    norm='l2',                   # 每行L2归一化
)

# 只对有效行计算
docs_valid = df.loc[valid_mask, '文档文本'].tolist()
commune_names_valid = df.loc[valid_mask, 'COL_C'].tolist()

tfidf_matrix = vectorizer.fit_transform(docs_valid)
vocab = vectorizer.get_feature_names_out()

# 还原实体名（把下划线换回空格）
vocab_display = [v.replace('_', ' ') for v in vocab]

print(f"词汇表大小（唯一实体数）：{len(vocab)}")

# ══════════════════════════════════════════════
# 4. 整理输出
# ══════════════════════════════════════════════

# ── ① TF-IDF 矩阵（公社 × 实体）──
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=commune_names_valid,
    columns=vocab_display
).round(6)
tfidf_df.index.name = 'COL_C（公社名）'

# ── ② 汇总统计 ──
# 词频（TF）：各实体在所有文档中出现的总次数
all_entities = []
for lst in df['实体列表']:
    all_entities.extend(lst)

from collections import Counter
entity_count = Counter(all_entities)

# 文档频次（DF）：含该实体的公社数
doc_freq = (tfidf_matrix > 0).sum(axis=0).A1

# IDF 值
N = valid_mask.sum()
idf_values = vectorizer.idf_

# 平均 TF-IDF（只在出现该词的文档中计算均值）
mean_tfidf_all   = tfidf_matrix.mean(axis=0).A1          # 所有文档均值
mean_tfidf_nonzero = np.array([
    tfidf_matrix[:, j].data.mean() if tfidf_matrix[:, j].nnz > 0 else 0
    for j in range(len(vocab))
])

summary_df = pd.DataFrame({
    '实体':             vocab_display,
    '总出现次数':        [entity_count.get(v.replace('_',' '), 0) for v in vocab],
    '出现公社数(DF)':    doc_freq.astype(int),
    'IDF值':            idf_values.round(4),
    '平均TF-IDF(全体)':  mean_tfidf_all.round(6),
    '平均TF-IDF(仅出现时)': mean_tfidf_nonzero.round(6),
}).sort_values('总出现次数', ascending=False).reset_index(drop=True)

summary_df['排名'] = summary_df.index + 1

# ══════════════════════════════════════════════
# 5. 打印 Top 20
# ══════════════════════════════════════════════
print("\n" + "="*65)
print("  Top 20 实体（按总出现次数排序）")
print("="*65)
print(f"{'排名':>4}  {'实体':<12}  {'总频次':>6}  {'公社数':>6}  "
      f"{'IDF':>6}  {'均值TF-IDF':>10}")
print("-"*65)
for _, row in summary_df.head(20).iterrows():
    print(f"{int(row['排名']):>4}  {row['实体']:<12}  "
          f"{int(row['总出现次数']):>6}  "
          f"{int(row['出现公社数(DF)']):>6}  "
          f"{row['IDF值']:>6.4f}  "
          f"{row['平均TF-IDF(全体)']:>10.6f}")
print("="*65)

# ══════════════════════════════════════════════
# 6. 写出 Excel
# ══════════════════════════════════════════════
print("\n写出结果文件...")

with pd.ExcelWriter(OUTPUT_SUMMARY, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='TF-IDF汇总', index=False)

    # 同时写一个按均值TF-IDF排序的版本
    summary_tfidf_sorted = summary_df.sort_values(
        '平均TF-IDF(全体)', ascending=False
    ).reset_index(drop=True)
    summary_tfidf_sorted['排名(按TF-IDF)'] = summary_tfidf_sorted.index + 1
    summary_tfidf_sorted.to_excel(writer, sheet_name='按TF-IDF排序', index=False)

print(f"✓ 汇总文件：{OUTPUT_SUMMARY}")

with pd.ExcelWriter(OUTPUT_MATRIX, engine='openpyxl') as writer:
    tfidf_df.to_excel(writer, sheet_name='TF-IDF矩阵')

print(f"✓ 矩阵文件：{OUTPUT_MATRIX}")

# ══════════════════════════════════════════════
# 7. 额外：按均值TF-IDF打印Top20（补充视角）
# ══════════════════════════════════════════════
print("\n" + "="*65)
print("  Top 20 实体（按平均TF-IDF排序，反映独特性）")
print("="*65)
print(f"{'排名':>4}  {'实体':<12}  {'总频次':>6}  {'公社数':>6}  "
      f"{'IDF':>6}  {'均值TF-IDF':>10}")
print("-"*65)
for _, row in summary_tfidf_sorted.head(20).iterrows():
    print(f"{int(row['排名(按TF-IDF)']):>4}  {row['实体']:<12}  "
          f"{int(row['总出现次数']):>6}  "
          f"{int(row['出现公社数(DF)']):>6}  "
          f"{row['IDF值']:>6.4f}  "
          f"{row['平均TF-IDF(全体)']:>10.6f}")
print("="*65)
print("\n完成！")

读取文件...
总行数：334，NER_实体非空：277 行
有效公社（含实体）：277 个

计算 TF-IDF...
词汇表大小（唯一实体数）：202

  Top 20 实体（按总出现次数排序）
  排名  实体               总频次     公社数     IDF    均值TF-IDF
-----------------------------------------------------------------
   1  米                105     105  1.9642    0.215776
   2  麦                 52      52  2.6573    0.121694
   3  小麦                36      36  3.0167    0.078377
   4  猪                 28      28  3.2603    0.058961
   5  绵                 27      27  3.2954    0.061771
   6  玉米                23      23  3.4496    0.038261
   7  蔬菜                22      22  3.4921    0.046211
   8  番薯                16      16  3.7944    0.030902
   9  大豆                15      15  3.8550    0.024663
  10  大麦                14      14  3.9196    0.025071
  11  鱼                  7       7  4.5482    0.013684
  12  绿肥                 7       7  4.5482    0.012456
  13  蚕                  7       7  4.5482    0.017139
  14  红薯                 6       6  4.6817    0.010409
  15  棉花

  7%|▋         | 5/75 [28:50:43<403:50:06, 20768.67s/it]


✓ 矩阵文件：E:\00论文\2025江苏人民公社\tfidf_matrix.xlsx

  Top 20 实体（按平均TF-IDF排序，反映独特性）
  排名  实体               总频次     公社数     IDF    均值TF-IDF
-----------------------------------------------------------------
   1  米                105     105  1.9642    0.215776
   2  麦                 52      52  2.6573    0.121694
   3  小麦                36      36  3.0167    0.078377
   4  绵                 27      27  3.2954    0.061771
   5  猪                 28      28  3.2603    0.058961
   6  蔬菜                22      22  3.4921    0.046211
   7  玉米                23      23  3.4496    0.038261
   8  番薯                16      16  3.7944    0.030902
   9  大麦                14      14  3.9196    0.025071
  10  大豆                15      15  3.8550    0.024663
  11  蚕                  7       7  4.5482    0.017139
  12  菜                  6       6  4.6817    0.016057
  13  粮食作物               5       5  4.8359    0.015305
  14  河泥                 5       5  4.8359    0.014939
  15  棉花                 6       